# SupplyMind AI — XGBoost

In [1]:
# -------------------
# Imports
# -------------------

from pathlib import Path

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)
from supplymind.features.predictions.ml.artifacts import save_model_artifact
from supplymind.features.predictions.ml.evaluation import (
    choose_threshold,
    evaluate_probabilities,
    positive_class_probability,
)
from supplymind.features.predictions.ml.preprocessing import build_preprocessor
from supplymind.features.predictions.ml.reporting import (
    save_evaluation_plots,
    save_feature_importance,
    save_json,
)
from supplymind.features.predictions.ml.training import (
    build_xgboost,
    fit_pipeline,
)
from supplymind.features.predictions.ml.workflow import (
    load_clean_syndelay,
    prepare_model_data,
)

In [2]:
# -------------------
# Project configuration
# -------------------

from pathlib import Path

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")
MODEL_ROOT = Path("../models")

assert DATASET_PATH.exists(), (
    f"Dataset not found at {DATASET_PATH}. "
    "Place syndelay_v1.csv under data/raw/syndelay/."
)

In [3]:
# -------------------
# Prepare identical model data
# -------------------

df = load_clean_syndelay(DATASET_PATH)
data = prepare_model_data(df)

In [4]:
# -------------------
# Build preprocessing
# -------------------

preprocessor = build_preprocessor(
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    scale_numerical=False,
)

In [5]:
# -------------------
# Train model
# -------------------

estimator = build_xgboost()
model = fit_pipeline(
    preprocessor,
    estimator,
    data.X_train,
    data.y_train,
)

In [6]:
# -------------------
# Validation probabilities
# -------------------

validation_probability = positive_class_probability(
    model,
    data.X_validation,
)

threshold, threshold_search = choose_threshold(
    data.y_validation,
    validation_probability,
)

print("Selected threshold:", threshold)
threshold_search.sort_values(
    ["f1", "recall"],
    ascending=False,
).head(10)

Selected threshold: 0.2700000000000001


,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,threshold
7,0.578828,0.578464,0.996434,0.731985,0.739064,0.834281,86,9775,48,13414,0.27
0,0.577241,0.577236,0.999851,0.731920,0.739064,0.834281,3,9858,2,13460,0.20
1,0.577156,0.577200,0.999703,0.731851,0.739064,0.834281,3,9858,4,13458,0.21
6,0.578099,0.577953,0.997400,0.731836,0.739064,0.834281,56,9805,35,13427,0.26
2,0.577198,0.577245,0.999480,0.731827,0.739064,0.834281,7,9854,7,13455,0.22
3,0.577284,0.577314,0.999257,0.731823,0.739064,0.834281,12,9849,10,13452,0.23
8,0.579299,0.578929,0.994354,0.731795,0.739064,0.834281,125,9736,76,13386,0.28
4,0.577284,0.577367,0.998663,0.731706,0.739064,0.834281,20,9841,18,13444,0.24
5,0.577327,0.577445,0.998069,0.731609,0.739064,0.834281,29,9832,26,13436,0.25
9,0.579814,0.579519,0.991235,0.731419,0.739064,0.834281,179,9682,118,13344,0.29


In [7]:
# -------------------
# Validation metrics
# -------------------

metrics = evaluate_probabilities(
    data.y_validation,
    validation_probability,
    threshold=threshold,
)

metrics.to_dict()

{'accuracy': 0.5788277665823436,
 'precision': 0.5784639268618742,
 'recall': 0.9964344079631555,
 'f1': 0.7319854847071021,
 'roc_auc': 0.7390639448578895,
 'average_precision': 0.8342811689796428,
 'true_negative': 86,
 'false_positive': 9775,
 'false_negative': 48,
 'true_positive': 13414,
 'threshold': 0.2700000000000001}

In [8]:
# -------------------
# Save candidate reports
# -------------------

MODEL_NAME = "xgboost"
REPORT_DIR = REPORT_ROOT / "models" / MODEL_NAME

save_json(
    metrics.to_dict(),
    REPORT_DIR / "validation_metrics.json",
)
threshold_search.to_csv(
    REPORT_DIR / "threshold_search.csv",
    index=False,
)
save_evaluation_plots(
    data.y_validation,
    validation_probability,
    threshold,
    REPORT_DIR,
    "validation",
)
save_feature_importance(
    model,
    REPORT_DIR / "feature_importance",
)

save_model_artifact(
    model,
    {
        "model_name": MODEL_NAME,
        "model_version": "0.1.0-candidate",
        "threshold": threshold,
        "validation_metrics": metrics.to_dict(),
    },
    MODEL_ROOT / "candidates" / MODEL_NAME,
)